# Part 1 — Scope, Ownership, and Outputs

### Objective

Build the analysis-ready MIMIC-III dataset for 48-hour AKI prediction. This notebook owns source-table validation, cohort construction, AKI labeling, landmark creation, feature extraction, and patient-level splitting.

### Rules

- Process all MIMIC data locally.
- Do not display, export, or commit patient-identifying or patient-level preview data.
- Use explicit configuration values and deterministic processing.
- Write versioned Parquet artifacts; do not pass notebook memory to downstream notebooks.

### Outputs

- ICU cohort table
- Landmark-level modeling dataset
- Patient-level split manifest
- Data dictionary and aggregate quality report


# Part 2 — Configuration and Reproducibility

### Objective

Load paths and study parameters from configuration and record enough metadata to reproduce the dataset.

### Required settings

- Local MIMIC-III root and artifact root
- Prediction landmarks: 8, 12, and 24 hours after ICU admission
- Prediction horizon: 48 hours
- Minimum required follow-up
- Feature lookback windows
- Split proportions and random seed
- AKI label variant and version

### Checks

Fail early when a required path, parameter, or source-table schema is missing.


# Part 3 — Load and Validate MIMIC-III Source Tables

### Objective

Load only the required columns from local MIMIC-III v1.4 tables and standardize identifiers, timestamps, units, and missing values.

### Inputs

Expected sources include PATIENTS, ADMISSIONS, ICUSTAYS, LABEVENTS, CHARTEVENTS, OUTPUTEVENTS, and the diagnosis, procedure, prescription, or intervention tables required by the approved feature and exclusion definitions.

### Rules

- Preserve subject_id, hadm_id, and icustay_id relationships.
- Record source row counts and timestamp ranges.
- Resolve duplicate and invalid measurements using documented rules.
- Never silently mix incompatible units.

### Output

Validated local source views or caches with documented schemas.


# Part 4 — Define the Base Study Population

### Objective

Create the base ICU cohort before landmark-specific eligibility is applied.

### Inclusion rules

- Adult patients according to the prespecified MIMIC age rule
- First ICU stay per patient
- Sufficient ICU follow-up for the study design

### Exclusion rules

- ESKD or chronic dialysis according to prespecified diagnosis and procedure definitions
- Invalid or unresolved ICU timing
- Other exclusions must be documented before use

### Output

One row per eligible ICU stay, keyed by subject_id, hadm_id, and icustay_id.


# Part 5 — Build the Creatinine Timeline and Baseline

### Objective

Create a clean, time-ordered serum creatinine trajectory for every eligible ICU stay and assign the prespecified prior baseline used by KDIGO.

### Rules

- Standardize creatinine units before comparison.
- Preserve measurement timestamps and provenance.
- Define how duplicate timestamps and implausible values are handled.
- Do not use measurements outside the permitted baseline definition.

### Checks

Report missing baselines, measurement counts, distributions, and manually review representative edge cases.


# Part 6 — Generate Creatinine-Based KDIGO Events

### Objective

Identify the earliest AKI onset and supporting measurements using the primary creatinine-based KDIGO definition.

### Definition

AKI is present when serum creatinine increases by at least 0.3 mg/dL within 48 hours or reaches at least 1.5 times the applicable prior baseline.

### Rules

- Evaluate measurement pairs chronologically.
- Store onset time, criterion, reference value, and current value.
- Make boundary inclusivity explicit.
- Do not define an onset earlier than the measurement that establishes the criterion.

### Validation cases

Include manually constructed positive, negative, boundary, duplicate-time, and missing-baseline cases.


# Part 7 — Construct Landmark Risk Sets and 48-Hour Targets

### Objective

Create prediction rows at 8, 12, and 24 hours after ICU admission.

### Temporal rules

- prediction_cutoff = ICU admission time + landmark
- feature_time <= prediction_cutoff
- label_window = (prediction_cutoff, prediction_cutoff + 48 hours]
- Patients with AKI at or before the cutoff are excluded from that landmark risk set.
- Adequate observation of the outcome window is required according to the study protocol.

### Output

One row per eligible icustay_id and landmark with cutoff time, outcome-window end, binary label, and optional onset time.


# Part 8 — Perform Cohort and Label Sanity Checks

### Objective

Verify that cohort and landmark counts are plausible before feature extraction.

### Reference counts

Previously observed creatinine-based results for adults, first ICU stay, and ICU stay of at least 72 hours were:

- 8-hour landmark: 11,878 analyzable; 3,044 AKI positive
- 12-hour landmark: 11,513 analyzable; 2,818 AKI positive
- 24-hour landmark: 10,522 analyzable; 2,083 AKI positive

These values are sanity-check targets, not hard-coded acceptance criteria. Any difference must be explained by the exact baseline, exclusion, time-boundary, or follow-up definitions.


# Part 9 — Extract Pre-Cutoff Features

### Objective

Extract demographics and time-window summaries from data available no later than each prediction cutoff.

### Candidate modalities

- Demographics
- Vital signs
- Laboratory measurements
- Glasgow Coma Scale
- Urine output
- Medications and interventions when feasible

### Time-series summaries

For each approved variable and lookback window, consider latest, mean, minimum, maximum, range, trend, count, and missingness.

### Leakage rule

Every contributing event timestamp must be less than or equal to the row's prediction cutoff. Post-cutoff urine may be used for a future label only, never as a feature.


# Part 10 — Integrate Features and Define the Dataset Schema

### Objective

Join all modalities into one landmark-level modeling table using explicit keys and stable feature names.

### Rules

- Keep raw identifiers separate from model features.
- Preserve subject_id, icustay_id, landmark, cutoff_time, label, and split linkage.
- Do not perform train-dependent imputation, scaling, or feature selection here.
- Distinguish true zeros from missing values.
- Maintain a machine-readable feature dictionary with source, unit, window, and aggregation.

### Output

A schema-validated landmark dataset with one row per eligible ICU stay and cutoff.


# Part 11 — Create Patient-Level Train, Validation, and Test Splits

### Objective

Create one reproducible split assignment per subject_id and attach it to every stay, landmark, and later snapshot belonging to that patient.

### Rules

- No subject_id may appear in more than one split.
- Use train for fitting.
- Use validation for hyperparameter tuning, feature decisions, calibration decisions, and alert-policy selection.
- Reserve test for locked final evaluation.
- Save the split manifest independently so every downstream analysis uses the same assignment.

### Checks

Verify patient disjointness, row counts, outcome prevalence, landmark distribution, and major subgroup balance.


# Part 12 — Export Artifacts and Dataset Quality Report

### Objective

Write versioned, local artifacts for the ML notebook and summarize dataset provenance without exposing patient-level information.

### Required metadata

- Dataset version and run identifier
- Configuration snapshot
- Source-table fingerprints or approved version identifiers
- Row and patient counts
- Feature schema
- Label prevalence by landmark and split
- Missingness summary
- Execution timestamp and Git commit

### Rule

Do not commit generated patient-level artifacts to GitHub.
